# 02 Data Cleaning

## Objective

Clean the raw CDC WONDER baseline mortality export and create a county-level processed dataset for analysis.

## Input

`data/raw/cdc_wonder_premature_mortality_2015_2019.csv`

## Output

`data/processed/baseline_premature_mortality_2015_2019_clean.csv`

## Key Cleaning Decisions

- Keep only rows with valid county FIPS codes.
- Convert county FIPS codes to five-digit strings.
- Split state abbreviation from county name.
- Convert mortality rates to numeric values.
- Preserve CDC WONDER unreliable-rate warnings as flag columns.


In [ ]:
import pandas as pd
from pathlib import Path

raw_path = Path('../data/raw/cdc_wonder_premature_mortality_2015_2019.csv')
processed_path = Path('../data/processed/baseline_premature_mortality_2015_2019_clean.csv')

raw = pd.read_csv(raw_path)
raw.head()

In [ ]:
raw.shape, raw.columns.tolist()

In [ ]:
# Keep only real county rows
df = raw[raw['County Code'].notna()].copy()

# Clean county FIPS code as a 5-digit string
df['county_fips'] = (
    df['County Code']
    .astype(int)
    .astype(str)
    .str.zfill(5)
)

# Split county/state
df['county'] = df['County']
df['state'] = df['County'].str.extract(r',\s*([A-Z]{2})$')

# Create unreliable flags before converting rates
df['crude_rate_unreliable_2015_2019'] = (
    df['Crude Rate'].astype(str).str.contains('Unreliable', case=False, na=False)
)

df['age_adjusted_rate_unreliable_2015_2019'] = (
    df['Age Adjusted Rate'].astype(str).str.contains('Unreliable', case=False, na=False)
)

# Convert rates to numeric; unreliable values become NaN
df['premature_mortality_crude_rate_2015_2019'] = pd.to_numeric(
    df['Crude Rate'], errors='coerce'
)

df['premature_mortality_age_adjusted_rate_2015_2019'] = pd.to_numeric(
    df['Age Adjusted Rate'], errors='coerce'
)

# Rename count/population columns
df = df.rename(columns={
    'Deaths': 'premature_deaths_2015_2019',
    'Population': 'population_2015_2019'
})

clean = df[[
    'county_fips',
    'county',
    'state',
    'premature_deaths_2015_2019',
    'population_2015_2019',
    'premature_mortality_crude_rate_2015_2019',
    'crude_rate_unreliable_2015_2019',
    'premature_mortality_age_adjusted_rate_2015_2019',
    'age_adjusted_rate_unreliable_2015_2019'
]].copy()

clean.head()

In [ ]:
# Validation checks
validation_summary = {
    'rows': len(clean),
    'duplicate_fips': clean['county_fips'].duplicated().sum(),
    'missing_county_fips': clean['county_fips'].isna().sum(),
    'missing_age_adjusted_rate': clean['premature_mortality_age_adjusted_rate_2015_2019'].isna().sum(),
    'unreliable_age_adjusted_rates': clean['age_adjusted_rate_unreliable_2015_2019'].sum()
}

validation_summary

In [ ]:
clean[[
    'premature_mortality_crude_rate_2015_2019',
    'premature_mortality_age_adjusted_rate_2015_2019'
]].describe()

In [ ]:
clean.to_csv(processed_path, index=False)
processed_path